# Can we predict next-month arrears?
Target: `arrears_next_month` (1 = account in arrears the month after the billing month). Data: one row per account-month, ~21k rows, 1,375 accounts, Jul 2023 - Dec 2025.

**Approach**
1. Check the data, especially anything that wouldn't be available at prediction time.
2. Validate **forward in time** (train on the past, test on later months). Random splits would mislead here: accounts repeat, and the arrears rate changes sharply over time.
3. Compare a simple baseline to logistic regression and gradient boosting.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 150)
df = pd.read_csv('data.csv', parse_dates=['billing_month']).sort_values(['meter_ref','billing_month']).reset_index(drop=True)
print(pd.read_csv('columns.csv').to_string(index=False))
print(df.shape, df.isna().sum().sum(), 'missing;', df.duplicated(['meter_ref','billing_month']).sum(), 'duplicate account-months')
df.describe().T

## 1. Target and a leakage check

In [ ]:
print('Overall arrears rate: %.3f' % df.arrears_next_month.mean())
df.groupby('final_notice_sent').arrears_next_month.agg(['mean','size'])

**`final_notice_sent` must not be used.** Per the data dictionary it is set for the billing cycle *that followed* the billing month, so it comes after the prediction point, alongside the outcome. It nearly restates the target (87% arrears when 1 vs 1.4% when 0). A model using it would look excellent offline and be useless in practice. It is excluded from all models; section 6 shows what it would do, for reference.

## 2. The arrears rate shifts strongly over time

In [ ]:
m = df.groupby('billing_month').arrears_next_month.mean()
fig, ax = plt.subplots(figsize=(9,3.5)); ax.plot(m.index, m.values, marker='o')
ax.set_ylabel('arrears rate'); ax.set_title('Arrears rate by billing month'); plt.show()
df['rate_eur_kwh'] = df.bill_eur / df.kwh
df.assign(q=df.billing_month.dt.to_period('Q')).groupby(['q','tariff']).rate_eur_kwh.mean().unstack().round(3)

The rate is about 2-4% until early 2025, then climbs to about 26% by Dec 2025. This coincides with a price rise (variable/prepay EUR/kWh rose from about 0.24 to 0.32 in early 2025; fixed stayed at 0.28), but the jump also hits fixed-tariff accounts, so price alone doesn't explain it, and the data can't identify the cause. Consequence: **the future base rate may differ from the training period**, so a model trained on old data will be miscalibrated, and honest evaluation must use the most recent months.

In [ ]:
df['late_bucket'] = df.late_payments_12m.clip(upper=3)
df['regime'] = np.where(df.billing_month >= '2025-04-01', 'Apr-Dec 2025', 'before Apr 2025')
for c in ['late_bucket','tariff','occupants']:
    print(df.pivot_table(index=c, columns='regime', values='arrears_next_month', aggfunc='mean').round(3), '\n')

`late_payments_12m` is the dominant signal in both regimes (the ordering holds, the level shifts). Tariff matters little; occupants shows a mild gradient.

## 3. Features
Only information available at the end of the billing month: tariff, occupants, kWh, bill, late payments, plus derived EUR/kWh, kWh per occupant, and change vs the same account's previous month. No calendar-time feature: trees can't extrapolate it and it would just memorise the regime.

In [ ]:
g = df.groupby('meter_ref')
df['kwh_per_occ'] = df.kwh / df.occupants
consec = g.billing_month.diff().dt.days.between(28, 31)
df['bill_chg'] = (df.bill_eur / g.bill_eur.shift() - 1).where(consec)
df['late_chg'] = (df.late_payments_12m - g.late_payments_12m.shift()).where(consec)
num = ['occupants','kwh','bill_eur','late_payments_12m','rate_eur_kwh','kwh_per_occ','bill_chg','late_chg']
cat = ['tariff']
print('rows with a consecutive previous month: %.2f' % consec.mean())

## 4. Time-based evaluation
Rolling origin: for each of the last 6 months, train on everything before it and test on that month. Then a main holdout: train through Sep 2025, test on Oct-Dec 2025. Metrics: ROC-AUC (ranking), PR-AUC (vs base rate), Brier (calibration); `mean_pred` vs `actual` shows level bias.

In [ ]:
def lr():
    return make_pipeline(ColumnTransformer([('n', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
                                            ('c', OneHotEncoder(), cat)]), LogisticRegression(max_iter=1000))
def gbm():
    return make_pipeline(ColumnTransformer([('n','passthrough',num), ('c', OneHotEncoder(sparse_output=False), cat)]),
                         HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))
def base():
    return make_pipeline(ColumnTransformer([('n','passthrough',['late_payments_12m'])]), LogisticRegression())
models = {'baseline: late_payments only': base, 'logistic regression': lr, 'gradient boosting': gbm}
def score(p, yt):
    return dict(auc=roc_auc_score(yt,p), pr_auc=average_precision_score(yt,p), brier=brier_score_loss(yt,p), mean_pred=p.mean(), actual=yt.mean())

months = sorted(df.billing_month.unique()); rows = []
for t in months[-6:]:
    tr_, te_ = df[df.billing_month < t], df[df.billing_month == t]
    for k, f in models.items():
        mdl = f().fit(tr_, tr_.arrears_next_month)
        rows.append(dict(test_month=str(t)[:7], model=k, **score(mdl.predict_proba(te_)[:,1], te_.arrears_next_month)))
roll = pd.DataFrame(rows)
print(roll.pivot(index='test_month', columns='model', values='auc').round(3))
roll.groupby('model')[['auc','pr_auc','brier','mean_pred','actual']].mean().round(3)

In [ ]:
cut = pd.Timestamp('2025-10-01'); tr, te = df[df.billing_month < cut], df[df.billing_month >= cut]
print(len(tr), len(te), '| train base rate %.3f, test base rate %.3f' % (tr.arrears_next_month.mean(), te.arrears_next_month.mean()))
res = []
for k, f in models.items():
    res.append(dict(model=k, **score(f().fit(tr, tr.arrears_next_month).predict_proba(te)[:,1], te.arrears_next_month)))
pd.DataFrame(res).set_index('model').round(3)

The models trained on all history under-predict the level (mean_pred well below actual) because most training data is from the low-arrears regime. Ranking (AUC) is less affected. Does training on the recent regime only fix the level?

In [ ]:
rec = tr[tr.billing_month >= '2025-04-01']; res = []; rec_fit = {}
for k, f in models.items():
    rec_fit[k] = f().fit(rec, rec.arrears_next_month)
    res.append(dict(model=k + ' [recent window]', **score(rec_fit[k].predict_proba(te)[:,1], te.arrears_next_month)))
pd.DataFrame(res).set_index('model').round(3)

In [ ]:
p = rec_fit['gradient boosting'].predict_proba(te)[:,1]
d_ = pd.DataFrame({'pred': p, 'y': te.arrears_next_month.values})
print(d_.groupby(pd.qcut(d_.pred, 10, duplicates='drop'), observed=True).mean().round(3))
o = np.argsort(-p); yt = te.arrears_next_month.values[o]
for frac in (.05, .1, .2, .3):
    n = int(len(p)*frac)
    print('top %2d%% by risk: %.0f%% in arrears, captures %.0f%% of all arrears' % (frac*100, yt[:n].mean()*100, yt[:n].sum()/yt.sum()*100))

## 5. What drives predictions?

In [ ]:
pi = permutation_importance(rec_fit['gradient boosting'], te[num + cat], te.arrears_next_month, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=num + cat).sort_values(ascending=False).round(4)

## 6. For reference: the leaky column (do not use)

In [ ]:
cols = ['final_notice_sent','late_payments_12m']
leak = LogisticRegression().fit(tr[cols], tr.arrears_next_month)
print('AUC with final_notice_sent: %.3f' % roc_auc_score(te.arrears_next_month, leak.predict_proba(te[cols])[:,1]))

## Conclusions
(See the tables above for numbers from this run.)

- **Partly yes.** Using only information available at the end of the billing month, the models rank accounts moderately (holdout AUC about 0.75 vs 0.70 for a late-payments-only baseline; not strong), driven almost entirely by `late_payments_12m`. kWh usage (and kWh per occupant) add a smaller amount; tariff, occupants and the month-on-month changes add essentially nothing.
- **Don't use `final_notice_sent`**: it is recorded after the prediction point and would give a falsely near-perfect result.
- **The arrears rate rose roughly 7x during 2025** and is still climbing, so models under-predict levels: training on all history gives a mean prediction of ~0.12 vs 0.25 actual, and even a recent-window model gives ~0.17. Ranking is more robust than absolute probabilities. Retrain often on recent data and monitor the base rate.
- **Caveats:** the cause of the 2025 shift is unknown; the new regime has only ~9 months and the holdout 3 months, so results are uncertain; accounts repeat, so rows aren't independent; I did not use the previous month's arrears status, which may be a strong feature if available at scoring time (worth checking with the business).
- **Suggested use:** prioritise outreach by predicted risk (see the top-percentile lift) rather than relying on exact probabilities, and validate on a fresh month before deploying.